# Notebook 3 (Hybrid): TTC Causal Attribution — Experiment 2
**IEEE IES GenAI Challenge 2026**

## What this notebook does

The LLM's role here is **causal attribution**, not classification.
The SVM already predicted a fault label. The 70B model is asked:

> *Given these sensor features and the predicted fault class, generate a
> step-by-step causal explanation of the physical degradation mechanism.
> Does the evidence support the prediction, or does it suggest a different fault?*

We sample N independent reasoning chains (temperature > 0), then apply
majority voting on the final fault attribution across N chains.

**Why this reduces API load dramatically:**
- Prompt is shorter (no 12 few-shot examples needed — the SVM label anchors context)
- Response is shorter (reasoning chain, not full classification)
- Token budget: ~800 tokens/call vs ~2200 before
- At Groq free 70B limit: ~7 calls/min safely → 200 × 10 = 2000 calls = ~5 hrs
- **Practical strategy:** run in 4 × 500-call sessions, cache persists between runs

**Metric:** Attribution accuracy = % of samples where majority-voted
LLM attribution matches ground truth label, measured at N = 1, 3, 5, 10.

**Key result to show:** attribution accuracy improves with N,
and LLM corrects some SVM misclassifications through causal reasoning.

In [ ]:
!pip install groq scikit-learn matplotlib seaborn tqdm -q

In [ ]:
import pickle, time, json, re, os
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from groq import Groq, RateLimitError
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

print('Ready.')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
GROQ_API_KEY  = 'your_groq_api_key_here'

MODEL         = 'llama-3.3-70b-versatile'   # non-negotiable
MAX_TOKENS    = 300      # shorter — attribution chain only, no few-shot block
TTC_TEMP      = 0.6      # diversity for hypothesis sampling
N_ITERATIONS  = [1, 3, 5, 10]
N_MAX         = 10

# Conservative rate for 70B on free tier
# 70B limit: ~6000 tokens/min input. Our prompt: ~500 tokens.
# Safe: 1 call per 6s = 10 RPM, well under limit.
CALL_DELAY    = 6.0      # seconds between API calls
MAX_RETRIES   = 5

LABEL_MAP    = {0: 'Normal', 1: 'Inner Race', 2: 'Outer Race', 3: 'Ball'}
NAME_TO_ID   = {v: k for k, v in LABEL_MAP.items()}
VALID_LABELS = list(LABEL_MAP.values())

CACHE_PATH = Path('ttc_cache.pkl')   # survives Colab disconnects

In [ ]:
with open('cwru_features.pkl', 'rb') as f:
    feat_data = pickle.load(f)

with open('baseline_results.pkl', 'rb') as f:
    baseline = pickle.load(f)

df_test  = baseline['df_test']   # already has svm_pred, svm_pred_name, svm_correct
fewshot  = feat_data['fewshot_examples']

print(f'Test samples:       {len(df_test)}')
print(f'SVM accuracy:       {baseline["accuracy"]*100:.2f}%')
print(f'SVM misclassified:  {baseline["n_misclassified"]} samples')
print(f'API calls needed:   {len(df_test)} × {N_MAX} = {len(df_test)*N_MAX}')
print(f'ETA per session:    {len(df_test)*N_MAX*CALL_DELAY/3600:.1f} hrs total '
      f'(run in batches, cache persists)')

## 1. Prompt Design

The key difference from the old approach: we give the model the SVM's predicted label
and ask it to *reason about causality*, not to classify from scratch.
This is shorter, faster, and scientifically cleaner — it isolates the attribution
reasoning contribution of the LLM from the classification contribution.

The model can either **confirm** the SVM prediction with physical reasoning,
or **override** it if the evidence points elsewhere. Override rate is itself
a useful metric — it measures how often LLM causal reasoning catches SVM errors.

In [ ]:
SYSTEM_PROMPT = """You are an expert in rotating machinery fault diagnosis with deep knowledge of vibration physics.

You will receive:
1. Vibration signal features from a CWRU bearing (12kHz, 1797 RPM)
2. A predicted fault class from an SVM classifier

Your task: perform CAUSAL ATTRIBUTION.
Reason step-by-step about whether the signal features physically support the SVM prediction,
or whether the evidence points to a different fault.

Physical reference:
- Normal:     kurtosis <3, crest factor <3, all band energies uniformly low
- Inner Race: kurtosis >4, crest factor >3.5, BPFI energy (147-177Hz) elevated
- Outer Race: moderate kurtosis 3-5, BPFO energy (92-122Hz) dominant
- Ball:       variable kurtosis, BSF energy (126-156Hz) dominant, often highest crest factor

Decision logic:
1. Assess impulsiveness: kurtosis and crest factor
2. Compare band energies: which of BPFI / BPFO / BSF is highest?
3. Check if band energy pattern is consistent with SVM prediction
4. If inconsistent, state the better-supported fault class

Respond ONLY with valid JSON, no preamble or explanation outside it:
{"fault_attribution": "<Normal|Inner Race|Outer Race|Ball>", "svm_agreement": <true|false>, "causal_chain": "<2-3 sentence step-by-step physical reasoning citing specific values>", "confidence": <0.0-1.0>}"""


def build_attribution_prompt(row):
    """Build a short attribution prompt using SVM prediction as context."""
    return (
        f"SVM predicted fault: {row['svm_pred_name']}\n"
        f"SVM confidence (probability): {row.get(f'svm_prob_{row["svm_pred_name"].replace(" ","_")}', 'N/A'):.3f}\n\n"
        f"Signal features:\n{row['text']}\n\n"
        f"Provide causal attribution:"
    )


def parse_attribution(text):
    """Parse LLM attribution response."""
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            ft = parsed.get('fault_attribution', '').strip()
            # Normalize
            for label in VALID_LABELS:
                if label.lower() in ft.lower():
                    ft = label
                    break
            if ft in VALID_LABELS:
                return {
                    'fault':       ft,
                    'agreement':   bool(parsed.get('svm_agreement', True)),
                    'causal_chain': parsed.get('causal_chain', ''),
                    'confidence':  float(parsed.get('confidence', 0.5)),
                }
    except Exception:
        pass
    # Fuzzy fallback
    for label in VALID_LABELS:
        if label.lower() in text.lower():
            return {'fault': label, 'agreement': True,
                    'causal_chain': text[:200], 'confidence': 0.4}
    return None


est_tokens = (len(SYSTEM_PROMPT) + 500) // 4
print(f'Estimated tokens per prompt: ~{est_tokens} (vs ~2200 in old approach)')
print(f'Token reduction: ~{(2200-est_tokens)/2200*100:.0f}%')

## 2. TTC Attribution Loop

**Batching strategy:** Run this cell once per session. It processes as many samples
as possible, saves the cache, and stops cleanly if interrupted. Re-run the next
session and it resumes from where it left off.

To limit a session to N samples: set `SESSION_LIMIT` below.

In [ ]:
# Set to None to run all remaining, or e.g. 200 to cap a session
SESSION_LIMIT = 200   # ~200 samples × 10 calls × 6s = 3.3 hrs per session

client = Groq(api_key=GROQ_API_KEY)

def call_groq(prompt_text, retries=MAX_RETRIES):
    delay = CALL_DELAY
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': prompt_text}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TTC_TEMP,
            )
            time.sleep(CALL_DELAY)
            return resp.choices[0].message.content
        except RateLimitError:
            wait = min(delay * (2 ** attempt), 180)
            print(f'  Rate limit — waiting {wait:.0f}s (attempt {attempt+1}/{retries})')
            time.sleep(wait)
        except Exception as e:
            wait = min(20 * (attempt+1), 60)
            print(f'  Error: {e} — waiting {wait}s')
            time.sleep(wait)
    return None


# ── Load or init cache ────────────────────────────────────────────────────────
if CACHE_PATH.exists():
    with open(CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    print(f'Cache loaded: {len(cache)} / {len(df_test)} samples complete.')
else:
    cache = {}
    print('No cache found — starting fresh.')

# Determine remaining samples
remaining = [idx for idx in df_test.index if idx not in cache]
if SESSION_LIMIT:
    remaining = remaining[:SESSION_LIMIT]

print(f'This session: {len(remaining)} samples × {N_MAX} hypotheses = '
      f'{len(remaining)*N_MAX} API calls')
print(f'ETA this session: ~{len(remaining)*N_MAX*CALL_DELAY/60:.0f} min')


# ── Main loop ─────────────────────────────────────────────────────────────────
for idx in tqdm(remaining, desc='TTC Attribution'):
    row    = df_test.loc[idx]
    prompt = build_attribution_prompt(row)

    hypotheses = []
    for n in range(N_MAX):
        raw = call_groq(prompt)
        if raw:
            parsed = parse_attribution(raw)
            if parsed:
                hypotheses.append(parsed)

    cache[idx] = {
        'hypotheses':    hypotheses,
        'true_label':    int(row['label']),
        'true_name':     row['label_name'],
        'svm_pred':      int(row['svm_pred']),
        'svm_pred_name': row['svm_pred_name'],
        'svm_correct':   bool(row['svm_correct']),
    }

    # Save checkpoint every 5 samples
    if len(cache) % 5 == 0:
        with open(CACHE_PATH, 'wb') as f:
            pickle.dump(cache, f)

# Final save
with open(CACHE_PATH, 'wb') as f:
    pickle.dump(cache, f)

total_done = len(cache)
total_all  = len(df_test)
print(f'\nSession complete. Cache: {total_done} / {total_all} samples.')
if total_done < total_all:
    print(f'{total_all - total_done} remaining — re-run this cell in next session.')

## 3. Evaluate — Run This Once Cache is Complete

You can run this cell with a partial cache to get interim results.
Results will be based on however many samples are cached.

In [ ]:
def majority_vote(hypotheses, n):
    """Majority vote over first n hypotheses. Returns winning label name or None."""
    votes = [h['fault'] for h in hypotheses[:n] if h and h.get('fault') in VALID_LABELS]
    if not votes:
        return None
    return Counter(votes).most_common(1)[0][0]


def evaluate_at_n(cache, n):
    """Compute attribution accuracy using first n hypotheses per sample."""
    true_ids, pred_ids = [], []
    for entry in cache.values():
        vote = majority_vote(entry['hypotheses'], n)
        if vote is None:
            continue
        true_ids.append(entry['true_label'])
        pred_ids.append(NAME_TO_ID[vote])
    if not true_ids:
        return None, None, [], []
    acc = accuracy_score(true_ids, pred_ids)
    f1  = f1_score(true_ids, pred_ids, average='macro', labels=[0,1,2,3],
                   zero_division=0)
    return acc, f1, true_ids, pred_ids


n_cached = len(cache)
print(f'Evaluating on {n_cached} cached samples...')
print(f'SVM baseline accuracy: {baseline["accuracy"]*100:.2f}%')
print()
print(f'{"N":>4} | {"TTC Accuracy":>14} | {"F1 Macro":>10} | {"vs SVM":>8}')
print('─' * 44)

ttc_results = {}
for n in N_ITERATIONS:
    acc, f1, true_l, pred_l = evaluate_at_n(cache, n)
    if acc is None:
        print(f'{n:>4} | insufficient data')
        continue
    delta = acc - baseline['accuracy']
    ttc_results[n] = {'accuracy': acc, 'f1': f1, 'true': true_l, 'pred': pred_l}
    print(f'{n:>4} | {acc*100:>13.2f}% | {f1:>10.4f} | {delta:>+7.2%}')

In [ ]:
# Override rate analysis — how often LLM disagrees with SVM
override_counts = {n: 0 for n in N_ITERATIONS}
override_correct = {n: 0 for n in N_ITERATIONS}  # overrides that were right

for entry in cache.values():
    svm_name = entry['svm_pred_name']
    true_name = entry['true_name']
    for n in N_ITERATIONS:
        vote = majority_vote(entry['hypotheses'], n)
        if vote and vote != svm_name:
            override_counts[n] += 1
            if vote == true_name:
                override_correct[n] += 1

print('\nLLM Override Analysis (disagreement with SVM):')
print(f'{"N":>4} | {"Overrides":>10} | {"Override Rate":>14} | {"Correct Overrides":>18}')
print('─' * 55)
for n in N_ITERATIONS:
    total = len(cache)
    ov    = override_counts[n]
    oc    = override_correct[n]
    print(f'{n:>4} | {ov:>10} | {ov/total*100:>13.1f}% | {oc:>10} ({oc/max(ov,1)*100:.0f}% of overrides correct)')

## 4. Plots

In [ ]:
if not ttc_results:
    print('No TTC results to plot yet — run more samples first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        'TTC Causal Attribution: Performance vs. Compute Iterations\n'
        f'(CWRU Bearing, Llama 3.3 70B, n={n_cached} samples)',
        fontsize=13, fontweight='bold'
    )

    ns       = [n for n in N_ITERATIONS if n in ttc_results]
    accs     = [ttc_results[n]['accuracy'] for n in ns]
    f1s      = [ttc_results[n]['f1'] for n in ns]
    svm_acc  = baseline['accuracy']
    svm_f1   = baseline['f1_macro']

    # ── Accuracy curve ────────────────────────────────────────────────────────
    ax = axes[0]
    ax.axhline(svm_acc, color='#888', linestyle='--', linewidth=1.5,
               label=f'SVM baseline ({svm_acc*100:.1f}%)')
    ax.plot(ns, accs, 'o-', color='#7B68EE', linewidth=2.5,
            markersize=9, label='TTC majority vote')
    for n, a in zip(ns, accs):
        ax.annotate(f'{a*100:.1f}%', (n, a),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9)
    ax.set_xlabel('TTC Hypotheses (N)', fontsize=11)
    ax.set_ylabel('Attribution Accuracy', fontsize=11)
    ax.set_title('Accuracy vs. TTC Iterations', fontsize=12)
    ax.set_xticks(ns)
    lo = min(svm_acc, min(accs)) - 0.05
    hi = max(svm_acc, max(accs)) + 0.05
    ax.set_ylim(max(0, lo), min(1.0, hi))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── F1 curve ─────────────────────────────────────────────────────────────
    ax2 = axes[1]
    ax2.axhline(svm_f1, color='#888', linestyle='--', linewidth=1.5,
                label=f'SVM baseline F1 ({svm_f1:.3f})')
    ax2.plot(ns, f1s, 's-', color='#2E8B57', linewidth=2.5,
             markersize=9, label='TTC F1 macro')
    for n, f in zip(ns, f1s):
        ax2.annotate(f'{f:.3f}', (n, f),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=9)
    ax2.set_xlabel('TTC Hypotheses (N)', fontsize=11)
    ax2.set_ylabel('F1 Score (Macro)', fontsize=11)
    ax2.set_title('F1 Score vs. TTC Iterations', fontsize=12)
    ax2.set_xticks(ns)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('ttc_accuracy_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved ttc_accuracy_curve.png')

In [ ]:
# Confusion matrix at best N
if ttc_results:
    best_n = max(ttc_results, key=lambda n: ttc_results[n]['accuracy'])
    r = ttc_results[best_n]
    cm_ttc = confusion_matrix(r['true'], r['pred'], labels=[0,1,2,3])
    target_names = [LABEL_MAP[i] for i in range(4)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Confusion Matrices: SVM Baseline vs. TTC Attribution', fontsize=13)

    sns.heatmap(baseline['confusion_matrix'], annot=True, fmt='d', cmap='Greys',
                xticklabels=target_names, yticklabels=target_names, ax=axes[0])
    axes[0].set_title(f'SVM Baseline\nAccuracy: {baseline["accuracy"]*100:.1f}%', fontsize=11)
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_ttc, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, ax=axes[1])
    axes[1].set_title(f'TTC Attribution (N={best_n})\nAccuracy: {r["accuracy"]*100:.1f}%', fontsize=11)
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

    plt.tight_layout()
    plt.savefig('ttc_vs_svm_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved ttc_vs_svm_confusion.png')

In [ ]:
# Sample reasoning chains for qualitative analysis in paper
print('=== SAMPLE TTC CAUSAL REASONING CHAINS ===')
shown = 0
for idx, entry in cache.items():
    if shown >= 4:
        break
    if not entry['hypotheses']:
        continue
    vote = majority_vote(entry['hypotheses'], N_MAX)
    if vote is None:
        continue
    correct   = '✓' if vote == entry['true_name'] else '✗'
    svm_agree = '(agrees with SVM)' if vote == entry['svm_pred_name'] else '(OVERRIDES SVM)'
    # Pick the hypothesis whose fault matches the majority vote for display
    rep = next((h for h in entry['hypotheses'] if h.get('fault') == vote), entry['hypotheses'][0])
    print(f'\n[{correct}] True: {entry["true_name"]:12s} | '
          f'SVM: {entry["svm_pred_name"]:12s} | '
          f'TTC(N=10): {vote:12s} {svm_agree}')
    print(f'Causal chain: {rep.get("causal_chain", "")[:350]}')
    shown += 1

## 5. Save Final Results

In [ ]:
final = {
    'ttc_results':      ttc_results,
    'baseline_accuracy': baseline['accuracy'],
    'baseline_f1':      baseline['f1_macro'],
    'n_iterations':     N_ITERATIONS,
    'n_cached':         n_cached,
    'override_counts':  override_counts,
    'override_correct': override_correct,
    'model':            MODEL,
    'ttc_temperature':  TTC_TEMP,
}

with open('ttc_results.pkl', 'wb') as f:
    pickle.dump(final, f)

print('Saved ttc_results.pkl')
print('Saved ttc_accuracy_curve.png')
print('Saved ttc_vs_svm_confusion.png')
print('\nM2 preliminary results ready.')